# Drone Chase Tag: PPO eğitimi

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** seç.
2. Hücreleri sırayla çalıştır. Sonuçlar Google Drive'a yazılır: oturum kapanırsa aynı hücreleri tekrar çalıştır,
   eğitim checkpoint'ten devam eder.
3. Eğitim log'u (`train_log.txt`), değerlendirme (`eval_*.json`) ve grafik (`curves.png`) koşu klasörüne
   otomatik kaydedilir. En sondaki **Arşiv** hücresi bunları zip olarak bilgisayarına indirir.

**Bu notebook `sp-faz1-s0` koşusuna ait.** Kod `42fc3f3` sürümüne sabitli; sonuçlar için `experiments/LOG.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")                        # hücre nerede çalışırsa çalışsın, önce sabit bir yere geç
REPO = "https://github.com/HasaNSaygili/DroneChaseTag.git"
if os.path.isdir("/content/proje/.git"):    # daha önce indirildiyse silmeden güncelle
    !git -C /content/proje fetch -q origin && git -C /content/proje reset -q --hard origin/main
else:
    !git clone -q {REPO} /content/proje
assert os.path.isfile("/content/proje/scripts/train.py"), "Kod indirilemedi: bu hücrenin çıktısını sohbete yapıştır"
os.chdir("/content/proje")
!git -C /content/proje checkout -q 42fc3f3   # bu koşunun kod sürümü
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

## Koşu ayarları
Aşağıdaki `RUNS` listesinde bütün koşular ayarlarıyla duruyor; `RUN`'a çalıştırmak istediğinin adını yaz.
Her koşunun tam ayarları ayrıca Drive'da `results/<RUN>/config.json` dosyasına yazılır.

In [ ]:
# Bütün koşular burada kayıtlı (adlar experiments/LOG.md ile aynı). Hiçbiri silinmez, yenisi alta eklenir.
# Her koşu: (script, ayarlar). "train" = tek rol sabit rakibe karşı, "selfplay" = iki rol sırayla.
# Çalıştırmak istediğin koşunun adını RUN'a yaz.
RUNS = {
    # 1) Kovalayan vs OPEN kaçanı, WCT kuralı → reward hacking (kaçanı rezonansla yere düşürdü)
    "ppo-chaser-empty-s0": ("train", ""),
    # 2) Aynısı, rakibi çarptırmak ödülsüz → kovalayan ebelemeyi öğrendi (%99,8)
    "ppo-chaser-empty-nocrashrew-s0": ("train", "opp_crash_reward=0.0"),
    # 3) Kaçan vs elle yazılmış kovalayan; γ 0.998 (kaçanın ödülü uzakta) → kaçan %99,8 dayandı
    "ppo-evader-empty-nocrashrew-s0": ("train", "learner=evader opponent=scripted_chaser opp_crash_reward=0.0 gamma=0.998"),
    # 4) Self-play, boş arena, WCT kuralı. Havuz 2 ve 3 numaralı koşuların politikalarıyla başlar. 20 nesil.
    #    → kovalayan baskın (kaçan en son rakibe karşı %0), kaçanın std'si 0,03'e çöktü
    "sp-empty-s0": ("selfplay", ""),
    # 5) Kontrol: 4'ün aynısı + entropi bonusu. Keşfin çökmesi tek başına dengesizliği açıklıyor mu?
    "sp-empty-ent-s0": ("selfplay", "ent_coef=0.003"),
    # 6) Asıl deney: faz 1 engeller (yerden tavana kutu + silindir), arena başına 8–12 (~%11 dolu) + entropi
    "sp-faz1-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.003'),
}
RUN = "sp-faz1-s0"
SCRIPT, SET = RUNS[RUN]
OUT = "/content/drive/MyDrive/DroneChaseTag/results"
print(RUN, "|", SCRIPT, "|", SET or "(varsayılan ayarlar)")

In [ ]:
import os
os.chdir("/content/proje")   # kurulum hücresi çalışmadıysa burada hata verir
!python -W ignore scripts/{SCRIPT}.py --name {RUN} --out {OUT} {('--set ' + SET) if SET else ''} 2>&1 | grep -v -E "warp|spec|fromstring"

## Değerlendirme ve video
Hiç görülmemiş arenalarda, gürültüsüz oynatır; 3 maçı videoya çeker (Drive'da `videos/`).

In [ ]:
import os
os.chdir("/content/proje")
if SCRIPT == "train":
    !MUJOCO_GL=egl python -W ignore scripts/watch_policy.py --run {OUT}/{RUN} --games 3 --eval-worlds 1024 --out /content/drive/MyDrive/DroneChaseTag/videos 2>&1 | grep -v -E "warp|spec|fromstring"
else:
    print("Self-play değerlendirmesi (Elo turnuvası ve videolar) ayrı bir adımda yapılacak.")

## Eğitim eğrisi

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f"{OUT}/{RUN}/metrics.jsonl")]
steps = np.array([r.get('steps', r.get('total_steps')) for r in rows]) / 1e6
C = np.array([r['outcome_counts'] for r in rows]); tot = C[:, 1:].sum(1, keepdims=True).clip(1)
names = ['ebeleme', 'süre doldu', 'kaçan çarptı', 'kovalayan çarptı']
K = 5   # kayan ortalama penceresi; 'valid': kenarlarda sahte düşüş olmasın diye uçlar atılır
smooth = lambda y: np.convolve(y, np.ones(K) / K, 'valid')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i, n in enumerate(names):
    ax[0].plot(steps[K // 2: len(steps) - K // 2], smooth(C[:, i + 1] / tot[:, 0]), label=n)
ax[0].set_title('sonuç oranları'); ax[0].set_xlabel('M adım'); ax[0].legend()
ax[1].plot(steps, [r['ep_len'] / 50 for r in rows]); ax[1].set_title('ortalama maç süresi (s)')
if 'gen' in rows[0]:   # self-play: öğrenenin en son rakibe karşı kazanması; nesil sınırları kesik çizgi
    ax[2].plot(steps, [r['win_latest_window'] for r in rows], label='en son rakibe karşı kazanma')
    for i in range(1, len(rows)):
        if rows[i]['gen'] != rows[i - 1]['gen']:
            for a in ax:
                a.axvline(steps[i], color='gray', lw=0.6, ls='--')
    ax[2].set_ylim(0, 1); ax[2].legend(); ax[2].set_title('self-play (nesiller kesik çizgiyle)')
else:
    ax[2].plot(steps, [r['explained_var'] for r in rows], label='explained var')
    ax[2].plot(steps, [r['std'] for r in rows], label='std'); ax[2].legend(); ax[2].set_title('critic ve keşif')
fig.suptitle(RUN)
fig.savefig(f"{OUT}/{RUN}/curves.png", dpi=120, bbox_inches="tight")   # koşu klasörüne kaydet
plt.show()

## Arşiv: koşunun dosyalarını bilgisayarına indir
config, log, ölçümler, değerlendirme, grafik ve son politika (checkpoint ve ara politikalar hariç: büyük)
tek bir zip olarak iner. Zip'i sohbette söyle; `experiments/<RUN>/` altına konup repoya eklenecek.
Başka bir koşuyu arşivlemek için aşağıda `ARCHIVE = "<koşu adı>"` yaz (Drive'da duran eski koşular için de çalışır).

In [ ]:
import glob, os, zipfile
from google.colab import files
ARCHIVE = RUN                      # eski bir koşu için: ARCHIVE = "ppo-chaser-empty-s0"
d = f"{OUT}/{ARCHIVE}"
keep = [p for p in glob.glob(f"{d}/*") if not p.endswith("checkpoint.pkl") and not os.path.basename(p).startswith("params_")]
params = sorted(glob.glob(f"{d}/params_*.pkl"))
keep += params[-1:]                # sadece son politika (makaledeki sayıların geldiği)
zpath = f"/content/{ARCHIVE}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, f"{ARCHIVE}/{os.path.basename(p)}")
print("\n".join(os.path.basename(p) for p in keep))
files.download(zpath)